# 05 · So sánh các mô hình và tổng hợp kết quả (Model Comparison & Synthesis)

Mục tiêu của notebook:
1. **Tổng hợp đa chiều (Multi-model Benchmark)**:
   - Thu thập kết quả đánh giá từ tất cả các kiến trúc: YOLOv8, YOLO11, Faster R-CNN, Mask R-CNN, CSRNet (và biến thể Tiling).
2. **So sánh Đánh đổi giữa Độ chính xác và Tốc độ (Accuracy vs Speed Trade-off)**:
   - Trực quan hóa tương quan giữa MAE vs FPS và MAE vs Số lượng tham số (Model Complexity).
3. **So sánh theo Từng họ kiến trúc (Paradigm Comparison)**:
   - Object Detection vs Instance Segmentation vs Density Regression.
   - Thử nghiệm thống kê (Wilcoxon signed-rank test / Paired t-test) xem sự khác biệt giữa các mô hình có mang ý nghĩa thống kê hay không.
4. **Trực quan hóa trực quan song song (Side-by-side Visual Comparison)**:
   - Hiển thị kết quả đếm (bounding box, segmentation mask, density map) của tất cả các mô hình trên cùng một vài ảnh mẫu thử nghiệm.
5. **Tự động xuất bảng Markdown & LaTeX**:
   - Xuất dữ liệu sẵn sàng để copy-paste trực tiếp vào báo cáo bài tập lớn hoặc bài báo khoa học.

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

# ---- Xác định thư mục gốc project: chạy được cả local lẫn Kaggle ----
IN_KAGGLE = Path('/kaggle/working').exists()
if IN_KAGGLE:
    CODE_INPUT = Path('/kaggle/input/fruitcount-code/FruitCount')   # sửa theo tên Kaggle Dataset chứa code
    ROOT = Path('/kaggle/working/FruitCount')
    if not ROOT.exists():
        shutil.copytree(CODE_INPUT, ROOT)
    OUT_BASE = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    while ROOT != ROOT.parent and not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent
    OUT_BASE = ROOT / 'outputs'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage

from src.utils import load_yaml, save_yaml, read_json, write_json, load_image, ensure_dir

DS_CFG_PATH = Path(os.environ.get('FRUITCOUNT_DATASET_CFG', ROOT / 'configs/dataset.yaml'))
DS_CFG = load_yaml(DS_CFG_PATH)
PROCESSED = ROOT / DS_CFG.get('processed_dir', 'data/processed')
RUNS = Path(os.environ.get('FRUITCOUNT_RUNS', OUT_BASE / 'runs' if IN_KAGGLE else ROOT / 'runs'))
OUT_DIR = ensure_dir(OUT_BASE / 'notebooks' / '05_compare_models')   # hình + bảng của notebook này

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
FAMILY_COLORS = {'detection': '#4C72B0', 'segmentation': '#55A868', 'density': '#C44E52'}


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


class StopExecution(Exception):
    """Dừng 'Run All' một cách êm (không in traceback)."""
    def _render_traceback_(self):
        return []


try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print(f'Môi trường : {"Kaggle" if IN_KAGGLE else "Local"}')
print(f'ROOT       : {ROOT}')
print(f'PROCESSED  : {PROCESSED}')
print(f'RUNS       : {RUNS}')
print(f'OUT_DIR    : {OUT_DIR}')

## 1. Cấu hình & Thu thập Kết quả

In [ ]:
COMPARE_SPLIT = 'test'
COMPARISON_OUT = RUNS / 'comparison'
COMPARISON_OUT.mkdir(parents=True, exist_ok=True)

from common import LOAD_REPORTS, bin_order

# Đọc kết quả từ tất cả các file metrics_<split>*.json
reports, per_image = {}, {}
for f in sorted(RUNS.glob(f'*/metrics_{COMPARE_SPLIT}*.json')):
    r = read_json(f)
    tag = f.stem[len(f'metrics_{COMPARE_SPLIT}'):]
    label = r['model'] + (' (tile)' if r.get('tiling') else '')
    r['_dir'], r['_tag'] = f.parent, tag
    reports[label] = r
    pi = f.parent / f'per_image_{COMPARE_SPLIT}{tag}.csv'
    if pi.exists():
        per_image[label] = pd.read_csv(pi, dtype={'image_id': str})

print(f"Thu thập được kết quả của {len(reports)} mô hình/cấu hình:")
for k, v in reports.items():
    print(f" - {k:25s} (Họ: {v['family']:12s}) | MAE: {v['counting']['mae']:.2f}")

if not reports:
    print("⚠️ Chưa có kết quả nào. Hãy chạy scripts/evaluate.py hoặc notebook 04_evaluate.ipynb trước!")

In [ ]:
import subprocess


def run(cmd, log_file=None):
    """Chạy lệnh con, in log trực tiếp (và ghi ra file nếu có). Trả về mã thoát."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    f = open(log_file, 'a', encoding='utf-8') if log_file else None
    try:
        p = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        for line in p.stdout:
            print(line, end='')
            if f:
                f.write(line)
        p.wait()
    finally:
        if f:
            f.close()
    if p.returncode != 0:
        print(f'⚠️ Lệnh kết thúc với mã {p.returncode}')
    return p.returncode

## 2. Bảng So sánh Tổng hợp (Master Comparison Table)
Trích xuất tất cả các chỉ số: MAE, RMSE, MAPE, $R^2$, mAP@0.5, mAP@[0.5:0.95], FPS, Latency và Params (M).

In [ ]:
COLUMNS = ["Model", "Family", "MAE", "RMSE", "MAPE (%)", "R2", "mAP@0.5", "mAP@[0.5:0.95]", "FPS", "Params (M)"]

table_rows = []
for name, r in reports.items():
    c = r.get('counting', {})
    d = r.get('detection', {})
    s = r.get('speed', {})
    table_rows.append({
        'Model': name,
        'Family': r.get('family', ''),
        'MAE': c.get('mae', np.nan),
        'RMSE': c.get('rmse', np.nan),
        'MAPE (%)': c.get('mape_pct', np.nan),
        'R2': c.get('r2', np.nan),
        'mAP@0.5': d.get('map50', np.nan),
        'mAP@[0.5:0.95]': d.get('map', np.nan),
        'FPS': s.get('fps', np.nan),
        'Params (M)': s.get('params_million', np.nan),
    })

df_comparison = pd.DataFrame(table_rows).sort_values('MAE').reset_index(drop=True)
df_comparison.to_csv(COMPARISON_OUT / f'comparison_{COMPARE_SPLIT}.csv', index=False)
display(df_comparison.round(3))

## 3. Biểu đồ Cột So sánh Đa chỉ số (Multi-metric Bar Charts)
So sánh trực quan MAE, RMSE, MAPE (%) giữa các mô hình. Màu sắc được phân loại theo họ kiến trúc.

In [ ]:
metrics_to_plot = ['MAE', 'RMSE', 'MAPE (%)']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5.5 * len(metrics_to_plot), 4.5))

colors = [FAMILY_COLORS.get(f, '#333333') for f in df_comparison['Family']]

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    bars = ax.barh(df_comparison['Model'], df_comparison[metric], color=colors, edgecolor='black', alpha=0.85)
    ax.invert_yaxis()
    ax.set_title(f"So sánh {metric}", fontweight='bold')
    ax.set_xlabel(metric)
    
    # Ghi giá trị số lên từng thanh
    for bar in bars:
        w = bar.get_width()
        if not np.isnan(w):
            ax.text(w + 0.02 * max(df_comparison[metric].dropna()), bar.get_y() + bar.get_height()/2,
                    f"{w:.2f}", va='center', fontsize=9)

# Thêm chú thích cho các họ mô hình
import matplotlib.patches as mpatches
legend_patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
axes[0].legend(handles=legend_patches, loc='lower right')

plt.tight_layout()
savefig(fig, "metrics_bar_comparison")

## 4. Đánh đổi Độ chính xác vs Tốc độ (Accuracy vs Speed & Parameters)
Mô hình lý tưởng nằm ở góc **dưới cùng bên phải** (MAE thấp nhất, FPS cao nhất).

In [ ]:
has_speed = df_comparison['FPS'].notna().any()

if has_speed:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 1. MAE vs FPS
    for _, row in df_comparison.iterrows():
        c = FAMILY_COLORS.get(row['Family'], '#333')
        if not np.isnan(row['FPS']):
            axes[0].scatter(row['FPS'], row['MAE'], color=c, s=120, edgecolors='black', zorder=4)
            axes[0].annotate(row['Model'], (row['FPS'] + 1, row['MAE'] + 0.05), fontsize=10)
    axes[0].set(title="Đánh đổi: Độ chính xác (MAE) vs Tốc độ (FPS)", xlabel="FPS (Khung hình/giây - Càng cao càng tốt)", ylabel="MAE (Càng thấp càng tốt)")
    axes[0].grid(True, linestyle='--', alpha=0.5)
    
    # 2. MAE vs Số tham số (Params)
    for _, row in df_comparison.iterrows():
        c = FAMILY_COLORS.get(row['Family'], '#333')
        if not np.isnan(row['Params (M)']):
            axes[1].scatter(row['Params (M)'], row['MAE'], color=c, s=120, edgecolors='black', zorder=4)
            axes[1].annotate(row['Model'], (row['Params (M)'] + 0.5, row['MAE'] + 0.05), fontsize=10)
    axes[1].set(title="Độ chính xác (MAE) vs Dung lượng mô hình (Triệu tham số)", xlabel="Số lượng tham số (M Params)", ylabel="MAE (Càng thấp càng tốt)")
    axes[1].grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    savefig(fig, "speed_accuracy_tradeoff")
else:
    print("Chưa có số liệu đo tốc độ FPS. Hãy chạy scripts/evaluate.py với cờ --speed để đo.")

## 5. So sánh Đường cong Sai số theo Mật độ Quả (Crowding Robustness Curves)
Tổng hợp đường cong sai số theo mật độ quả từ tất cả các mô hình lên cùng một biểu đồ để trả lời câu hỏi cốt lõi của đề tài:
*Khi số quả tăng cao, Object Detection hay Density Regression kiểm soát sai số tốt hơn?*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for name, r in reports.items():
    bin_data = r.get('error_by_count_bin', [])
    if bin_data:
        df_b = pd.DataFrame(bin_data)
        df_b['sort_key'] = df_b['gt_count_bin'].apply(lambda x: int(str(x).replace('>=', '').split('-')[0]))
        df_b = df_b.sort_values('sort_key')
        
        c = FAMILY_COLORS.get(r['family'], '#333')
        style = '--' if 'tile' in name else '-'
        ax.plot(df_b['gt_count_bin'], df_b['mae'], marker='s', linestyle=style, color=c, lw=2.2, label=f"{name}")

ax.set(
    title=f"Khả năng chống chịu mật độ dày đặc: Sai số MAE theo phân khúc số quả ({COMPARE_SPLIT.upper()})",
    xlabel="Số lượng quả thực tế trên ảnh (Ground Truth Bin)",
    ylabel="Sai số tuyệt đối trung bình (MAE)"
)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
savefig(fig, "density_robustness_comparison")

## 6. Kiểm định Thống kê Ý nghĩa Sai số (Statistical Significance Test)
Sử dụng kiểm định Wilcoxon signed-rank trên sai số tuyệt đối $|Pred - GT|$ của từng ảnh giữa mô hình tốt nhất với các mô hình còn lại.
Giá trị $p < 0.05$ khẳng định sự vượt trội có ý nghĩa thống kê chứ không phải do ngẫu nhiên.

In [ ]:
try:
    from scipy.stats import wilcoxon
    
    if len(per_image) >= 2:
        best_m = df_comparison.iloc[0]['Model']
        best_errors = per_image[best_m].set_index('image_id')['abs_error']
        
        stat_rows = []
        for m_name, df_pi in per_image.items():
            if m_name == best_m:
                continue
            other_errors = df_pi.set_index('image_id')['abs_error']
            common_ids = best_errors.index.intersection(other_errors.index)
            
            diff = other_errors.loc[common_ids] - best_errors.loc[common_ids]
            res = wilcoxon(diff, alternative='greater')
            
            stat_rows.append({
                'So sánh': f"{best_m} vs {m_name}",
                'Chênh lệch MAE': float(other_errors.loc[common_ids].mean() - best_errors.loc[common_ids].mean()),
                'Wilcoxon Stat': float(res.statistic),
                'p-value': float(res.pvalue),
                'Ý nghĩa thống kê (p < 0.05)': 'Có' if res.pvalue < 0.05 else 'Không'
            })
        
        df_stat = pd.DataFrame(stat_rows)
        display(df_stat.round(4))
    else:
        print("Cần ít nhất 2 mô hình có dữ liệu per_image để thực hiện kiểm định thống kê.")
except ImportError:
    print("Chưa cài scipy — bỏ qua kiểm định thống kê.")

## 7. Trực quan hóa Song song Kết quả Dự đoán (Side-by-side Visuals)
Lựa chọn một ảnh có mật độ quả cao và trực quan hóa kết quả dự đoán của tất cả các mô hình lên cạnh nhau:
- Detection (YOLO / Faster R-CNN): Bounding boxes
- Segmentation (Mask R-CNN): Instance masks
- Density (CSRNet): Heatmap mật độ

In [ ]:
from src.data import load_split_records
from src.inference import CountingPredictor
from src.visualization import render_result, draw_count_banner

# Lấy 2 ảnh từ tập test (1 ảnh mật độ trung bình, 1 ảnh mật độ rất đông)
test_recs = load_split_records(PROCESSED, COMPARE_SPLIT)
test_recs_sorted = sorted(test_recs, key=lambda r: r.count)
selected_samples = [
    test_recs_sorted[len(test_recs_sorted)//2], # trung bình
    test_recs_sorted[-1]                        # đông nhất
]

# Tải predictors cho các mô hình cơ bản (không tile)
unique_base_models = list({r['model']: r for r in reports.values()}.values())
loaded_predictors = {}

for r in unique_base_models:
    try:
        p = CountingPredictor.from_config(ROOT / f"configs/models/{r['model']}.yaml", weights=r['weights'])
        loaded_predictors[r['model']] = p
    except Exception as e:
        print(f"Không thể nạp {r['model']}: {e}")

fig, axes = plt.subplots(len(selected_samples), len(loaded_predictors) + 1,
                         figsize=(5 * (len(loaded_predictors) + 1), 5 * len(selected_samples)), squeeze=False)

for row_idx, sample in enumerate(selected_samples):
    img = load_image(sample.image_path)
    
    # Cột 0: Ground Truth
    from src.visualization import draw_boxes
    gt_vis = draw_boxes(img, sample.boxes)
    gt_vis = draw_count_banner(gt_vis, f"Ground Truth: {sample.count}")
    axes[row_idx][0].imshow(gt_vis)
    axes[row_idx][0].set_title(f"GT ({sample.image_id})", fontsize=11, fontweight='bold')
    axes[row_idx][0].axis('off')
    
    # Các cột tiếp theo: Dự đoán của từng model
    for col_idx, (m_name, pred_obj) in enumerate(loaded_predictors.items(), start=1):
        res = pred_obj.predict_array(img)
        pred_vis = render_result(img, res)
        axes[row_idx][col_idx].imshow(pred_vis)
        axes[row_idx][col_idx].set_title(f"{m_name} (Pred: {res.count:.1f})", fontsize=11)
        axes[row_idx][col_idx].axis('off')

plt.tight_layout()
savefig(fig, "visual_comparison_side_by_side")

## 8. Tự động Xuất Định dạng Báo cáo (Markdown & LaTeX)
Lưu bảng tổng hợp ra file Markdown và file LaTeX để chèn trực tiếp vào báo cáo.

In [ ]:
# Xuất Markdown
md_content = df_comparison.round(3).to_markdown(index=False)
(COMPARISON_OUT / f'comparison_table_{COMPARE_SPLIT}.md').write_text(md_content, encoding='utf-8')

# Xuất LaTeX
latex_content = df_comparison.round(3).to_latex(index=False, caption="Bảng so sánh tổng hợp các mô hình đếm trái cây", label="tab:fruitcount_benchmark")
(COMPARISON_OUT / f'comparison_table_{COMPARE_SPLIT}.tex').write_text(latex_content, encoding='utf-8')

print("Đã lưu bảng tổng kết tại:")
print(" -", COMPARISON_OUT / f'comparison_table_{COMPARE_SPLIT}.md')
print(" -", COMPARISON_OUT / f'comparison_table_{COMPARE_SPLIT}.tex')

display(Markdown("### Bảng Markdown sẵn sàng cho Báo cáo:\n" + md_content))

## 9. Kết luận Tổng thể Đề tài
- Đã xây dựng và đánh giá thành công hệ thống đếm trái cây đa kiến trúc (**FruitCount**).
- Việc chia split đồng nhất và chuẩn hóa `CountResult` cho phép so sánh công bằng và khách quan giữa **Object Detection**, **Instance Segmentation** và **Density Regression**.
- Đã làm rõ câu hỏi nghiên cứu về sự tương quan giữa mật độ quả và độ chính xác của từng phương pháp.